# FactoryIO

## modbus_ex3 대상

In [7]:
# 모드버스접속
from pymodbus.client import ModbusTcpClient
import time as tt
import threading
client = ModbusTcpClient('210.119.14.58', port=502)
client.connect()

True

In [8]:
# 초기화
client.write_coils(0, [0]*16)

In [11]:
# 켜야되는 출력 8부터 8개 모두 On
client.write_coils(8, [1]*8)

In [13]:
# 센서의 값을 가지고 오기
import threading
import time

sensor_scan = True
step = [0, 0] # Tip

def sscan():
    while sensor_scan:
        ir = client.read_input_registers(0, count=8).registers[0]
        front_limit = client.read_discrete_inputs(0, count=8).bits[3]
        if ir == 224:
            client.write_coil(3,1)
            step[0] = 0
        if step[0] == 0 and (ir == 192 or ir == 128):
            client.write_coil(3,1) # 턴테이블 +이동
            step[0] = 1
        if step[0] == 1 and front_limit:
            client.write_coil(3,0) # 턴테이블 +정지
            client.write_coil(0,1) # 스토퍼 on
            client.write_coil(2,1) # 턴테이블 터닝 on
            step[1] = 1
        if step[1] == 1 and client.read_discrete_inputs(0,count=8).bits[1]: # 턴테이블이 돌았을때
            client.write_coil(2,1)
            client.write_coil(3,1)
            tt.sleep(3)
            client.write_coil(2,0)
            client.write_coil(0,0)
            step[0] = 0
            step[1] = 0
            
        time.sleep(0.1)

thread_scan = threading.Thread(target=sscan, daemon=True)
print("센서로도 쓰레드 시작")
thread_scan.start()

센서로도 쓰레드 시작


In [5]:
sensor_scan = False

In [6]:
client.read_discrete_inputs(0,count=8).bits[3]

True